In [ ]:
# !pip --version

pip 26.2.1 from D:\yugyeong\hanwha_0902\days\ex0914\.venv\Lib\site-packages\pip (python 3.12)



In [23]:
# 랭체인 설치 및 -U: 이미 설치되어 있으면 업데이트
# langchain 랭체인 핵심 기능, langchain-openai OpenAI 랭체인에서 사용, 
# %pip install -U langchain langchain-openai

In [28]:
from dotenv import load_dotenv
import os

# .env 파일 로드
load_dotenv()

# 정상 로드 확인 테스트
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

OpenAI 키: sk-proj-...
LANGSMITH 키: lsv2_pt_...
LangSmith 프로젝트: test0914


In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model= "gpt-4o-mini")

question = "일본의 수도는 어디인가요?"

response = llm.invoke(question)

print(response.content)

일본의 수도는 도쿄입니다.


In [43]:
# OpenAI llm 가져오기(두뇌)
from langchain_openai import ChatOpenAI

# Pyhon 함수 Agent의 도구로 만들기(도구)
from langchain_core.tools import tool

# LLM + Tool 묶어서 Agent 만들기(두뇌 + 도구 연결해서 Agent 구성)
from langchain.agents import create_agent

# 1. Tool 정의
@tool
# (city는 문자열 타입 힌트) -> 문자열을 반환 타입 힌트 
# city > str > get_weather() > str 
def get_weather(city: str) -> str:
    # 함수 설명, agent Tool 용도 설명 
    """Get weather for a given city."""
    
    # 입력받은 도시 이름을 넣어서 문자열을 반환하는 예제
    return f"lt`s always sunny in {city}"

@tool
def find_restaurant(city: str) -> str:
    """Find restaurants in a city."""
    return f"{city}의 맛집을 찾았습니다."

# 2. 모델 및 에이전트 연결
# 사용할 LLM(두뇌)을 생성, model: 사용할 모델, temperature: 답변 다양성 설정(낮을수록 일관, 안정적)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# tools=[get_weather] Agent가 사용할 수 있는 Tool 등록
agent = create_agent(model=llm, tools=[get_weather, find_restaurant], system_prompt="Always answer in Korean.")

# 3. 실행
# Agent에게 입력으로 전달하고 invoke()를 통해 Agent 실행
result = agent.invoke({"messages": [("user", "천안 맛집")]})
print(result["messages"][-1].content) 


천안의 맛집을 찾았습니다. 어떤 종류의 음식을 원하시나요? 예를 들어 한식, 중식, 양식 등 특정한 종류가 있으신가요?


In [ ]:
# HumanMessage 사용자 입력, AIMessage AI가 판단하거나 최종 답변을 생성(Tool을 사용하라고 요청할 수도 있음)
# ToolMessage Tool 실행 결과, AIMessage Tool 결과를 보고 만든 최종 답변
for i, message in enumerate(result["messages"]):
    print(i, type(message).__name__)
    print(message)
    print("----------------")
    

0 HumanMessage
content='Seoul' additional_kwargs={} response_metadata={} id='25568c2e-d4ef-475e-b228-399cb2417fc2'
----------------
1 AIMessage
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 72, 'total_tokens': 119, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c19ce61615', 'id': 'chatcmpl-ENvFY2xQcWrwEg0IM4kffT2aNAfBN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a09ec8-43da-7080-a6d2-d480ce262ce3-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Seoul'}, 'id': 'call_IZ6LpgxNeG5mDlH2de8mWap6', 'type': 'tool_call'}, {'

In [ ]:
# TeddyNote에서 제공하는 LangSmith 추적 기능
# !pip install langchain_teddynote
# from langchain_teddynote import logging

# LangSmit 공식 추적 방식

# 추적 켜기
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "test0914"

# 추적 끄기
# os.environ["LANGSMITH_PROJECT"] = "test0914"
# os.environ["LANGSMITH_TRACING"] = "false"